### ЗАДАЧА: Сборка конфигураций сервисов для релиза

DevOps-команда готовит вечерний релиз нескольких сервисов.
У каждого сервиса много параметров: профиль запуска, число реплик, мониторинг,
автоскейлинг, canary-выкатка и переменные окружения.

Если собирать такие конфиги длинными словарями вручную,
код быстро становится неудобным и легко ошибиться.
Поэтому в этой задаче нужно использовать паттерн `Builder`.

НЕОБХОДИМО:

1. Реализовать класс `ServiceConfigBuilder`.
2. Внутри билдера хранить текущую конфигурацию сервиса.
3. Реализовать цепочку методов:
   - `set_service(name, image)`
   - `for_api()`
   - `for_worker()`
   - `for_highload()`
   - `enable_monitoring()`
   - `enable_autoscaling(min_replicas, max_replicas)`
   - `enable_canary(percent)`
   - `add_env(key, value)`
   - `build()`
4. На основе `release_jobs` собрать список итоговых конфигураций `release_configs`.
5. После сборки дополнительно посчитать:
   - список сервисов с canary-выкаткой,
   - сервис с максимальным числом реплик,
   - количество сервисов с включенным мониторингом.
6. Вывести все собранные конфигурации и итоговую сводку.


In [6]:
release_jobs = [
    {
        "name": "billing-api",
        "image": "registry.example.com/billing:1.4.0",
        "profile": "api",
        "monitoring": True,
        "autoscaling": (2, 6),
        "canary_percent": 10,
        "env": {"MODE": "prod", "REGION": "eu-central"},
    },
    {
        "name": "email-worker",
        "image": "registry.example.com/email-worker:2.1.3",
        "profile": "worker",
        "monitoring": True,
        "autoscaling": None,
        "canary_percent": 0,
        "env": {"QUEUE": "emails", "RETRIES": "5"},
    },
    {
        "name": "feed-generator",
        "image": "registry.example.com/feed-generator:3.0.1",
        "profile": "highload",
        "monitoring": True,
        "autoscaling": (4, 12),
        "canary_percent": 0,
        "env": {"BATCH_SIZE": "5000", "REGION": "global"},
    },
    {
        "name": "admin-api",
        "image": "registry.example.com/admin:0.9.8",
        "profile": "api",
        "monitoring": False,
        "autoscaling": None,
        "canary_percent": 25,
        "env": {"MODE": "prod", "ACCESS_LEVEL": "internal"},
    },
]


class ServiceConfigBuilder:
    def __init__(self):
        # TODO 1: создайте словарь self.config со значениями по умолчанию:
        #   service_name -> None
        #   image -> None
        #   replicas -> 1
        #   cpu_limit -> 1
        #   memory_gb -> 1
        #   monitoring -> False
        #   autoscaling -> None
        #   canary_percent -> 0
        #   env -> пустой словарь
        self.config = {
            'service_name':None,
            'image':None,
            'replicas':1,
            'cpu_limit':1,
            'memory_gb':1,
            'monitoring':False,
            'autoscaling':None,
            'canary_percent':0,
            'env':{}
        }

    def set_service(self, name: str, image: str):
        # TODO 2: сохраните name и image в self.config
        self.config['service_name'] = name
        self.config['image'] = image
        # TODO 3: верните self
        return self
        

    def for_api(self):
        # TODO 4: настройте профиль API:
        #   replicas = 2
        #   cpu_limit = 1
        #   memory_gb = 2
        self.config['replicas'] = 2
        self.config['cpu_limit'] = 1
        self.config['memory_gb'] = 2
        # TODO 5: верните self
        return self
        

    def for_worker(self):
        # TODO 6: настройте профиль worker:
        #   replicas = 1
        #   cpu_limit = 2
        #   memory_gb = 4
        self.config['replicas'] = 1
        self.config['cpu_limit'] = 2
        self.config['memory_gb'] = 4

        # TODO 7: верните self
        return self
        

    def for_highload(self):
        # TODO 8: настройте highload-профиль:
        #   replicas = 4
        #   cpu_limit = 4
        #   memory_gb = 8
        self.config['replicas'] = 4
        self.config['cpu_limit'] = 4
        self.config['memory_gb'] = 8

        # TODO 9: верните self
        return self
        

    def enable_monitoring(self):
        # TODO 10: включите monitoring
        self.config['monitoring'] = True
        # TODO 11: верните self
        return self
        

    def enable_autoscaling(self, min_replicas: int, max_replicas: int):
        # TODO 12: сохраните autoscaling как словарь:
        #   {"enabled": True, "min": min_replicas, "max": max_replicas}
        self.config['autoscaling'] = {"enabled": True, "min": min_replicas, "max": max_replicas}
        # TODO 13: верните self
        return self

    def enable_canary(self, percent: int):
        # TODO 14: сохраните canary_percent
        self.config['canary_percent'] = percent
        # TODO 15: верните self
        return self
        

    def add_env(self, key: str, value: str):
        # TODO 16: добавьте пару key/value в env
        self.config['env'][key] = value
        # TODO 17: верните self
        return self

    def build(self):
        # TODO 18: верните копию итоговой конфигурации
        # Подсказка: env и autoscaling тоже лучше копировать отдельно
        import copy
        return copy.deepcopy(self.config)
        


release_configs = []
services_with_canary = []
max_replicas_service = None
max_replicas = 0
monitoring_enabled_count = 0

for job in release_jobs:
    builder = ServiceConfigBuilder()
    builder.set_service(job["name"], job["image"])

    # TODO 19: выберите профиль по job['profile']:
    #   api -> for_api()
    #   worker -> for_worker()
    #   highload -> for_highload()
    if job['profile'] == 'api':
        builder.for_api()
    if job['profile'] == 'worker':
        builder.for_worker()
    if job['profile'] == 'highload':
        builder.for_highload()

    # TODO 20: если monitoring == True, вызовите enable_monitoring()
    if job['monitoring'] == True:
        builder.enable_monitoring()

    # TODO 21: если autoscaling не None,
    # распакуйте кортеж и вызовите enable_autoscaling(min_replicas, max_replicas)
    if job['autoscaling'] is not None:
        min_replicas, max_replicas = job['autoscaling']
        builder.enable_autoscaling(min_replicas, max_replicas)
    # TODO 22: если canary_percent > 0, вызовите enable_canary(...)
    if job['canary_percent'] > 0:
        builder.enable_canary(job['canary_percent'])
    # TODO 23: добавьте все переменные окружения из job['env'] через add_env()
    for key, value in job['env'].items():
        builder.add_env(key, value)

    config = builder.build()
    release_configs.append(config)

    # TODO 24: если в конфиге canary_percent > 0, добавьте service_name в services_with_canary
    if job['canary_percent'] > 0:
        services_with_canary.append(job['name'])

    # TODO 25: если monitoring включен, увеличьте monitoring_enabled_count
    if job['monitoring'] is True:
        monitoring_enabled_count += 1
    # TODO 26: если replicas больше текущего max_replicas,
    # обновите max_replicas и max_replicas_service
    if config['replicas'] > max_replicas:
        max_replicas = config['replicas']
        max_replicas_service = config['service_name']
        max_replicas_service = job['name']
        max_replicas = job['replicas']

print("Собранные конфигурации релиза:")
for config in release_configs:
    print(config)

print()
print("Сервисы с canary:", services_with_canary)
print("Сервис с максимальным числом реплик:", max_replicas_service, max_replicas)
print("Количество сервисов с мониторингом:", monitoring_enabled_count)


Собранные конфигурации релиза:
{'service_name': 'billing-api', 'image': 'registry.example.com/billing:1.4.0', 'replicas': 2, 'cpu_limit': 1, 'memory_gb': 2, 'monitoring': True, 'autoscaling': {'enabled': True, 'min': 2, 'max': 6}, 'canary_percent': 10, 'env': {'MODE': 'prod', 'REGION': 'eu-central'}}
{'service_name': 'email-worker', 'image': 'registry.example.com/email-worker:2.1.3', 'replicas': 1, 'cpu_limit': 2, 'memory_gb': 4, 'monitoring': True, 'autoscaling': None, 'canary_percent': 0, 'env': {'QUEUE': 'emails', 'RETRIES': '5'}}
{'service_name': 'feed-generator', 'image': 'registry.example.com/feed-generator:3.0.1', 'replicas': 4, 'cpu_limit': 4, 'memory_gb': 8, 'monitoring': True, 'autoscaling': {'enabled': True, 'min': 4, 'max': 12}, 'canary_percent': 0, 'env': {'BATCH_SIZE': '5000', 'REGION': 'global'}}
{'service_name': 'admin-api', 'image': 'registry.example.com/admin:0.9.8', 'replicas': 2, 'cpu_limit': 1, 'memory_gb': 2, 'monitoring': False, 'autoscaling': None, 'canary_perce